# Chapter 36 — Large Language Models: Tokens, Decoding, Scaling

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, warnings, re; warnings.filterwarnings("ignore")
from collections import Counter, defaultdict

def make_corpus(n_docs, seed):
    """Synthetic customer-review-style text from a hand-built grammar with
    realistic word frequencies (Zipfian), so tokenization, decoding, and
    scaling can be demonstrated on text with no copyright status question."""
    r = np.random.default_rng(seed)
    subjects = ["the product", "this item", "the device", "the service", "delivery",
                "the packaging", "the battery", "the screen", "support", "the app"]
    verbs = ["arrived", "worked", "broke", "improved", "failed", "lasted",
             "shipped", "performed", "exceeded", "disappointed"]
    adverbs = ["quickly", "slowly", "eventually", "immediately", "barely",
               "consistently", "rarely", "surprisingly", "finally", "never"]
    objects = ["expectations", "the first week", "two days", "a month",
               "the warranty period", "every test", "the price point",
               "my needs", "the description", "the competition"]
    connectors = ["and", "but", "so", "because", "although", "while"]

    def weighted(options, alpha=1.3):
        w = np.array([1 / (i + 1) ** alpha for i in range(len(options))])
        return r.choice(options, p=w / w.sum())

    docs = []
    for _ in range(n_docs):
        n_sent = r.integers(1, 4)
        sentences = []
        for _ in range(n_sent):
            s = f"{weighted(subjects)} {weighted(verbs)} {weighted(adverbs)}"
            if r.random() < 0.6:
                s += f" {weighted(connectors)} {weighted(subjects)} {weighted(verbs)} {weighted(objects)}"
            sentences.append(s)
        docs.append(". ".join(sentences) + ".")
    return docs

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
# Byte-pair encoding builds a vocabulary bottom-up: start from individual
# characters, and repeatedly merge whichever adjacent pair appears most
# often, treating that pair as one new unit from then on.
def get_pair_counts(word_freqs):
    pairs = Counter()
    for word, freq in word_freqs.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_pair(pair, word_freqs):
    merged = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, freq in word_freqs.items():
        merged[word.replace(bigram, replacement)] = freq
    return merged

corpus = make_corpus(200, seed=36)
text = " ".join(corpus)
words = text.split()
word_freqs = Counter(" ".join(list(w)) + " </w>" for w in words)     # start as characters

print(f"corpus: {len(words)} word occurrences, {len(word_freqs)} distinct words")
print(f"starting vocabulary: {len(set(c for w in word_freqs for c in w.split()))} characters")

merges = []
vocab_over_time = []
wf = dict(word_freqs)
for step in range(30):
    pairs = get_pair_counts(wf)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    wf = merge_pair(best, wf)
    merges.append((best, pairs[best]))
    vocab_size = len(set(c for w in wf for c in w.split()))
    vocab_over_time.append(vocab_size)

print(f"\nfirst eight merges, most frequent adjacent pair each round:")
for i, (pair, count) in enumerate(merges[:8], 1):
    print(f"  {i}. {pair[0]!r} + {pair[1]!r}  (seen together {count} times)")

### Block 2  (`c2.py`)

In [ ]:
# A larger vocabulary means fewer tokens per sentence, at the cost of a
# larger table of units the model must represent and predict over.
def apply_merges(text, merges):
    tokens = list(text) 
    words_with_boundary = []
    for w in text.split(" "):
        words_with_boundary.append(list(w) + ["</w>"])
    for pair, _ in merges:
        for wi, w in enumerate(words_with_boundary):
            new_w, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == pair[0] and w[i+1] == pair[1]:
                    new_w.append(w[i] + w[i+1]); i += 2
                else:
                    new_w.append(w[i]); i += 1
            words_with_boundary[wi] = new_w
    return [tok for w in words_with_boundary for tok in w]

test_sentence = "the product arrived quickly and this item worked consistently."

print(f"{'merges applied':>15}{'vocabulary size':>18}{'tokens for one sentence':>26}")
for n_merges in (0, 5, 10, 20, 30):
    partial_merges = merges[:n_merges]
    toks = apply_merges(test_sentence, partial_merges)
    vocab = len(set(c for w in wf for c in w.split())) if n_merges == len(merges) else None
    vocab_size = 26 + n_merges + 1          # 26 letters + merges + boundary marker, roughly
    print(f"{n_merges:>15}{vocab_over_time[n_merges-1] if n_merges else 27:>18}"
          f"{len(toks):>26}")

print(f"\nthe sentence: {test_sentence!r}")
print(f"at 0 merges (characters):  {apply_merges(test_sentence, [])}")
print(f"at 20 merges:              {apply_merges(test_sentence, merges[:20])}")

### Block 3  (`c3.py`)

In [ ]:
# A next-token distribution to decode from. A trigram model: predict the
# next word from the two words before it, estimated by counting.
def train_ngram(docs, n=3):
    counts = defaultdict(Counter)
    for doc in docs:
        words = ["<s>"] * (n - 1) + doc.split() + ["</s>"]
        for i in range(len(words) - n + 1):
            context = tuple(words[i:i + n - 1])
            counts[context][words[i + n - 1]] += 1
    return counts

def next_word_probs(counts, context, vocab, smoothing=0.1):
    c = counts.get(context, Counter())
    total = sum(c.values()) + smoothing * len(vocab)
    return {w: (c.get(w, 0) + smoothing) / total for w in vocab}

train_docs = make_corpus(400, seed=36)
vocab = sorted(set(w for d in train_docs for w in d.split()) | {"</s>"})
counts = train_ngram(train_docs, n=3)

context = ("the", "product")
probs = next_word_probs(counts, context, vocab)
top5 = sorted(probs.items(), key=lambda x: -x[1])[:5]
print(f"trained on {len(train_docs)} synthetic reviews, vocabulary of {len(vocab)} words")
print(f"\nafter the context {context}, most likely next words:")
for w, p in top5:
    print(f"  {w:<14}{p:.4f}")
print(f"\ntotal probability mass: {sum(probs.values()):.4f}   (must sum to 1)")

### Block 4  (`c4.py`)

In [ ]:
# Four ways to turn a probability distribution into an actual next word.
def sample_from(probs_dict, method, seed, temperature=1.0, k=5, p=0.9):
    r = np.random.default_rng(seed)
    words = list(probs_dict.keys())
    p_arr = np.array([probs_dict[w] for w in words])

    if method == "greedy":
        return words[p_arr.argmax()]
    if method == "temperature":
        logp = np.log(p_arr + 1e-12) / temperature
        adj = np.exp(logp - logp.max()); adj /= adj.sum()
        return r.choice(words, p=adj)
    if method == "top_k":
        idx = np.argsort(-p_arr)[:k]
        sub = p_arr[idx]; sub /= sub.sum()
        return r.choice([words[i] for i in idx], p=sub)
    if method == "top_p":
        order = np.argsort(-p_arr)
        cum = np.cumsum(p_arr[order])
        cutoff = np.searchsorted(cum, p) + 1
        idx = order[:cutoff]
        sub = p_arr[idx]; sub /= sub.sum()
        return r.choice([words[i] for i in idx], p=sub)

def generate(counts, vocab, method, seed, max_len=12, **kw):
    r_start = np.random.default_rng(seed)
    words = ["<s>", "<s>"]
    for i in range(max_len):
        context = tuple(words[-2:])
        probs = next_word_probs(counts, context, vocab)
        nxt = sample_from(probs, method, seed=seed * 1000 + i, **kw)
        if nxt == "</s>":
            break
        words.append(nxt)
    return " ".join(words[2:])

print("greedy (always the single most likely word):")
print(" ", generate(counts, vocab, "greedy", seed=36))
print(" ", generate(counts, vocab, "greedy", seed=37))
print(" ", generate(counts, vocab, "greedy", seed=38))

print("\ntemperature = 0.7 (sample, mildly reshaped toward the top):")
for s in (36, 37, 38):
    print(" ", generate(counts, vocab, "temperature", seed=s, temperature=0.7))

print("\ntop-k = 3 (sample among the three most likely words only):")
for s in (36, 37, 38):
    print(" ", generate(counts, vocab, "top_k", seed=s, k=3))

print("\ntop-p = 0.9 (sample among the smallest set covering 90% mass):")
for s in (36, 37, 38):
    print(" ", generate(counts, vocab, "top_p", seed=s, p=0.9))

### Block 5  (`c5.py`)

In [ ]:
# Quantify what "repetitive" actually means: the fraction of generated
# bigrams that are exact repeats of an earlier bigram in the same
# generation, averaged over many independent generations.
def repetition_rate(counts, vocab, method, n_runs=40, **kw):
    rates = []
    for i in range(n_runs):
        text = generate(counts, vocab, method, seed=1000 + i, max_len=20, **kw)
        toks = text.split()
        if len(toks) < 3:
            continue
        bigrams = list(zip(toks, toks[1:]))
        seen, repeats = set(), 0
        for b in bigrams:
            if b in seen:
                repeats += 1
            seen.add(b)
        rates.append(repeats / max(len(bigrams), 1))
    return np.mean(rates)

print(f"{'method':<16}{'mean bigram repetition rate':>30}")
for method, kw in [("greedy", {}), ("temperature (0.7)", {"temperature": 0.7}),
                   ("top-k (3)", {"k": 3}), ("top-p (0.9)", {"p": 0.9})]:
    key = "temperature" if "temperature" in method else ("top_k" if "top-k" in method else
          ("top_p" if "top-p" in method else "greedy"))
    rate = repetition_rate(counts, vocab, key, **kw)
    print(f"{method:<16}{rate:>30.4f}")

### Block 6  (`c6.py`)

In [ ]:
# Scaling laws, at a scale small enough to run in seconds rather than
# GPU-months: does more training data reduce a language model's error,
# and does that benefit run out? Perplexity is the standard metric,
# the exponential of the average negative log-likelihood per word.
def perplexity(counts, vocab, docs, n=3):
    total_logp, total_words = 0.0, 0
    for doc in docs:
        words = ["<s>"] * (n - 1) + doc.split() + ["</s>"]
        for i in range(n - 1, len(words)):
            context = tuple(words[i - n + 1:i])
            probs = next_word_probs(counts, context, vocab)
            total_logp += np.log(probs.get(words[i], 1e-12))
            total_words += 1
    return np.exp(-total_logp / total_words)

held_out = make_corpus(300, seed=999)             # fixed evaluation set, never trained on
vocab_full = sorted(set(w for d in (train_docs + held_out) for w in d.split()) | {"</s>"})

print(f"{'training documents':>19}{'perplexity on held-out text':>30}")
for n_docs in (10, 30, 100, 300, 1000, 3000):
    docs_n = make_corpus(n_docs, seed=36)
    counts_n = train_ngram(docs_n, n=3)
    ppl = perplexity(counts_n, vocab_full, held_out)
    print(f"{n_docs:>19}{ppl:>30.2f}")

print(f"\nfor reference, a model assigning equal probability to all ")
print(f"{len(vocab_full)} words would score a perplexity of {len(vocab_full)}.")

### Block 7  (`c7.py`)

In [ ]:
# Capacity interacts with data. A higher-order model (more context,
# more parameters in the count table) can represent more, but each
# additional context needs its own examples: too little data and a
# larger model overfits to noise in its own counts.
print(f"{'training docs':>14}{'bigram (n=2)':>14}{'trigram (n=3)':>15}{'4-gram (n=4)':>14}")
for n_docs in (10, 50, 300, 2000):
    docs_n = make_corpus(n_docs, seed=36)
    row = [n_docs]
    for order in (2, 3, 4):
        counts_n = train_ngram(docs_n, n=order)
        ppl = perplexity(counts_n, vocab_full, held_out, n=order)
        row.append(ppl)
    print(f"{row[0]:>14}{row[1]:>14.2f}{row[2]:>15.2f}{row[3]:>14.2f}")

print(f"\nmore context helps once there is enough data to fill it in;")
print(f"with too little data, the extra context has nothing reliable")
print(f"to have learned, and can score worse than the simpler model.")